# 07 — Spatial features (Tier 5)

Adds **state** (categorical, one-hot) to the engineered features parquet.

Latitude/longitude are already features. State captures unobserved regional factors:
- variety popularity (TX vs NE wheat varieties differ)
- management practices (irrigation prevalence)
- typical sowing dates

Source: Natural Earth `ne_50m_admin_1_states_provinces.shp` (cartopy bundle)

**Output**: `features.parquet` with state + state one-hot columns.

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import warnings
warnings.filterwarnings('ignore')

FEAT_IN  = 'data/processed/features/features_with_windowed.parquet'
FEAT_OUT = 'data/processed/features/features.parquet'
STATES_SHP = 'data/external/ne_50m_admin_1_states_provinces.shp'

feat = pd.read_parquet(FEAT_IN)
feat['field_id'] = feat['field_id'].astype(str)
feat['year'] = feat['year'].astype(int)
print(f'Features (input): {feat.shape}')

In [ ]:
# Spatial join with US states
states = gpd.read_file(STATES_SHP)
us = states[states['admin']=='United States of America'][['postal','geometry']].rename(columns={'postal':'state'})

gdf = gpd.GeoDataFrame(feat, geometry=gpd.points_from_xy(feat['longitude'], feat['latitude']), crs='EPSG:4326')
gdf = gdf.sjoin(us, how='left', predicate='within').drop(columns=['index_right','geometry'])
feat_with_state = pd.DataFrame(gdf)

print('State distribution:')
print(feat_with_state['state'].value_counts(dropna=False))

# Fill missing state (edge cases) with nearest state (not strictly correct but rare)
if feat_with_state['state'].isna().any():
    print(f'WARNING: {feat_with_state["state"].isna().sum()} rows with NaN state')
    feat_with_state['state'] = feat_with_state['state'].fillna('UNK')

In [ ]:
# One-hot encode state (drop first to avoid collinearity for linear models)
state_dummies = pd.get_dummies(feat_with_state['state'], prefix='state', drop_first=False).astype(int)
print(f'State one-hot cols: {state_dummies.columns.tolist()}')
feat_with_state = pd.concat([feat_with_state, state_dummies], axis=1)
print(f'Features (with state): {feat_with_state.shape}')

feat_with_state.to_parquet(FEAT_OUT, index=False)
print(f'Saved: {FEAT_OUT}')